In [10]:
"""
Authors: Kayla Huang, Thomas Xu Zhang, and Wuhao Cao
Investigating our big question: What are good choices of the line search parameters?
"""

'\nAuthors: Kayla Huang, Thomas Xu Zhang, and Wuhao Cao\nInvestigating our big question: What are good choices of the line search parameters?\n'

# Overview

## Objectives and Scope
We decided to investigate the bigger question of what are good choices of the line search parameters. 
To accomplish this, we will systematically test (1) different Armijo line search parameters (c1 and c2) and (2) backtracking line search parameters (alpha, tau, c1). We will evaluate these parameters across the different given problems, using the different methods to find the moving directions. We will utilize a broad range of parameter values that includes both common chosen values and extreme bounds to observe their respective convergence speed failure rates across different algorithms and problems.


## Methodology and Evaluation Metric
To determine the optimal parameter configurations and answer our big question, we will benchmark our algorithms using the following approach:
- Methods Tested: We will test our parameter grid on the Gradient Descent, modified Newton, BFGS Newton, and DFPNewton algorithms using backtracking line search and Wolfe line search methods, respectively.
- Test Suite: We will execute these configurations across the provided suite of twelve test problems.
- Evaluation Metrics: For each run in our parameter grid, we will extract and record the number of iterations, function evaluations, gradient evaluations, and CPU seconds required to solve the problem.
- Comparative Analysis: We will analyze the resulting metrics and plot the function value versus iteration plots, to identify which parameter choices are the most robust and efficient.

In [7]:
import numpy as np
from types import SimpleNamespace
from optSolver import optSolver_Fire_Horse
import project_problems
import algorithms

# Method Testing

In [10]:
# Test parameter grid on the Gradient Descent, modified Newton, BGFS Newton, and DFPNewton
# Do for each of the 12 problems, and for each of the 4 algorithms, and for each of the line search methods (Armijo, backtracking)
# For each run, extract and record the number of iterations, function evaluations, gradient evaluations, and CPU seconds

# Define the line search parameters to test
line_search_params = {
    'Armijo': {
        'c1': [1e-4, 1e-3, 1e-2],
        'alpha0': [1.0, 0.5, 0.1]
    },
    'Backtracking': {
        'rho': [0.5, 0.8, 0.9],
        'alpha0': [1.0, 0.5, 0.1]
    }
}
# Define the optimization algorithms to test
algorithms = ['Gradient Descent', 'Modified Newton', 'BGFS Newton', 'DFP Newton']
# Define the problems to test
problems = [project_problems.quad_10_10_func, project_problems.quad_10_1000_func, 
            project_problems.quad_1000_10_func, project_problems.quad_1000_1000_func, 
            project_problems.quartic_1_func, project_problems.quartic_2_func, 
            project_problems.rosenbrock_2_func, project_problems.rosenbrock_100_func, 
            project_problems.data_fit_2_func, project_problems.exp_10_func, 
            project_problems.exp_100_func, project_problems.genhumps_5_func]

# Define x0 for each problem
problem_x0 = {
    'quad_10_10_func': np.zeros(10),
    'quad_10_1000_func': np.zeros(10),
    'quad_1000_10_func': np.zeros(1000),
    'quad_1000_1000_func': np.zeros(1000),
    'quartic_1_func': np.zeros(4),
    'quartic_2_func': np.zeros(4),
    'rosenbrock_2_func': np.array([-1.2, 1.0]),
    'rosenbrock_100_func': np.concatenate([[-1.2, 1.0], np.zeros(98)]),
    'data_fit_2_func': np.zeros(2),
    'exp_10_func': np.zeros(10),
    'exp_100_func': np.zeros(100),
    'genhumps_5_func': np.zeros(5)
}

# Initialize a results dictionary to store the outcomes
results = {}
# Loop through each problem, algorithm, and line search method
for problem_func in problems:
    prob_name = problem_func.__name__
    x0 = problem_x0[prob_name]
    # Manuallyspecify the function, gradient,and hessian using the functions in project_problems.py
    grad_func = getattr(project_problems, prob_name.replace('_func', '_grad'))
    hess_func = getattr(project_problems, prob_name.replace('_func', '_Hess'))
    # set up the problem object for the optimization solver
    problem = SimpleNamespace(
        name=prob_name,
        x0=x0,
        compute_f=problem_func,
        compute_g=grad_func,
        compute_H=hess_func
    )
    for algorithm in algorithms:
        for line_search_method, params in line_search_params.items():
            # Map algorithm names to match the code
            method_name = algorithm.replace(' ', '')
            if method_name == 'BGFSNewton':
                method_name = 'BFGS'
            elif method_name == 'DFPNewton':
                method_name = 'DFP'
            
            # Loop through the parameter combinations for the current line search method
            if line_search_method == 'Armijo':
                for c1 in params['c1']:
                    for alpha0 in params['alpha0']:
                        # Create method and options objects
                        method = SimpleNamespace(
                            name=method_name,
                            options={
                                "step_type": "Backtracking",  # Armijo uses backtracking in the code
                            }
                        )
                        options = SimpleNamespace(
                            term_tol=1e-6,
                            max_iterations=1000,
                            c1=c1,
                            backtracking_alpha=alpha0,
                            tau=0.5  # default rho
                        )
                        # Run the optimization solver
                        print(f"Running {algorithm} on {prob_name} with {line_search_method} (c1={c1}, alpha0={alpha0})")
                        result = optSolver_Fire_Horse(problem, method, options)
                        # Extract and record the results
                        key = (prob_name, algorithm, line_search_method, f'c1={c1}', f'alpha0={alpha0}')
                        results[key] = {
                            'iterations': result.iterations,
                            'function_evaluations': result.function_evaluations,
                            'gradient_evaluations': result.gradient_evaluations,
                            'cpu_seconds': result.cpu_seconds
                        }
            elif line_search_method == 'Backtracking':
                for rho in params['rho']:
                    for alpha0 in params['alpha0']:
                        # Create method and options objects
                        method = SimpleNamespace(
                            name=method_name + 'W',  # Assuming W for Wolfe
                            options={
                                "step_type": "Wolfe",
                            }
                        )
                        options = SimpleNamespace(
                            term_tol=1e-6,
                            max_iterations=1000,
                            c=rho,  # rho as c
                            initial_alpha=alpha0,
                            c1=1e-4,  # default
                            c2=0.9   # default
                        )
                        # Run the optimization solver
                        result = optSolver_Fire_Horse(problem, method, options)
                        # Store results
                        key = (prob_name, algorithm, line_search_method, f'rho={rho}', f'alpha0={alpha0}')
                        results[key] = {
                            'iterations': result.iterations,
                            'function_evaluations': result.function_evaluations,
                            'gradient_evaluations': result.gradient_evaluations,
                            'cpu_seconds': result.cpu_seconds
                        }
# After running all combinations, we can analyze the results to identify good choices of line search parameters
# For example, we can look for parameter combinations that consistently yield low iterations and CPU time across multiple problems and algorithms 

print("Line Search Parameter Tuning Results:")
for key, value in results.items():
    print(f"{key}: {value}")   

Running Gradient Descent on quad_10_10_func with Armijo (c1=0.0001, alpha0=1.0)


TypeError: only length-1 arrays can be converted to Python scalars

# Comparative Analysis

In [ ]:
# Analyze the resulting metrics and plot the function value vs. iteration plots